# Exploring the solar wind

In this notebook we will use measurements of the solar wind near Earth to explore three questions:

1. What is the solar wind like **right now**?
2. What counts as typical or unusual?
3. Can we recognize large-scale structure and recurrence in the solar wind?

Code for loading, selecting, and plotting the data is provided. Change time intervals or variables when you want to investigate something new, and record your observations in the Markdown cells.

> Work in groups. There may be data gaps. Hopefully the solar wind is not boring today...

## Data used in this notebook

### Recent data

The most recent 24 hours come from NOAA's real-time solar-wind service. We select **SOLAR-1**, the spacecraft currently providing NOAA's primary operational measurements from the Sun–Earth L1 region. 

These are operational real-time data. They may contain gaps or values that are later revised. They are measurements near L1.

- [NOAA real-time solar-wind page](https://www.spaceweather.gov/products/solar-wind)
- [NOAA SOLAR-1 overview](https://www.nesdis.noaa.gov/our-satellites/future-programs/swfo/space-weather-observations-l1-advance-readiness-solar-1)
- [Magnetic-field JSON](https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json)
- [Plasma JSON](https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json)

### Longer record

For longer intervals we use NASA's **OMNI** dataset. OMNI combines measurements from several upstream spacecraft and time-shifts them to the nose of Earth's bow shock. We access the hourly record through NASA's Heliophysics API (HAPI).

- [NASA OMNI documentation](https://omniweb.gsfc.nasa.gov/html/ow_data.html)
- [NASA CDAWeb HAPI server](https://cdaweb.gsfc.nasa.gov/hapi)

In [ ]:
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True})

# 1. The solar wind during the last 24 hours

The cell below downloads the magnetic-field and plasma files, selects `SOLAR1`, and combines samples whose times differ by no more than 90 seconds. It returns a Pandas DataFrame indexed by time.

Units used below:

| Quantity | Unit |
|---|---|
| Proton speed | km/s |
| Proton density | cm⁻³ |
| Proton temperature | K |
| Magnetic field | nT |

In [ ]:
MAG_URL = "https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json"
WIND_URL = "https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json"


def download_recent_solar_wind():
    mag = pd.read_json(MAG_URL)
    wind = pd.read_json(WIND_URL)

    mag = mag[mag.source == "SOLAR1"].copy()
    wind = wind[wind.source == "SOLAR1"].copy()

    mag["time"] = pd.to_datetime(mag.pop("time_tag"), utc=True)
    wind["time"] = pd.to_datetime(wind.pop("time_tag"), utc=True)

    mag = mag[[
        "time", "source", "active", "bt",
        "bx_gse", "by_gse", "bz_gse",
        "bx_gsm", "by_gsm", "bz_gsm", "overall_quality"
    ]].sort_values("time")

    wind = wind[[
        "time", "proton_speed", "proton_density",
        "proton_temperature", "overall_quality"
    ]].sort_values("time")

    combined = pd.merge_asof(
        mag, wind, on="time", direction="nearest",
        tolerance=pd.Timedelta("90s"),
        suffixes=("_mag", "_plasma")
    )

    combined = combined.rename(columns={
        "bt": "Bmag",
        "bx_gse": "Bx_GSE", "by_gse": "By_GSE", "bz_gse": "Bz_GSE",
        "bx_gsm": "Bx_GSM", "by_gsm": "By_GSM", "bz_gsm": "Bz_GSM",
        "proton_speed": "speed",
        "proton_density": "density",
        "proton_temperature": "temperature"
    })

    return combined.set_index("time").sort_index()

In [ ]:
solar1 = download_recent_solar_wind()

print(f"Spacecraft label in file: {solar1['source'].iloc[0]}")
print(f"First sample: {solar1.index.min()}")
print(f"Latest sample: {solar1.index.max()}")
print(f"Number of magnetic-field samples: {len(solar1)}")

solar1.tail()

In [ ]:
def plot_solar_wind_overview(frame, title="Solar wind"):
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(frame.index, frame["speed"], color="tab:blue")
    axes[0].set_ylabel("Speed\n[km/s]")

    axes[1].plot(frame.index, frame["density"], color="tab:orange")
    axes[1].set_ylabel("Density\n[cm$^{-3}$]")

    axes[2].plot(frame.index, frame["temperature"], color="tab:red")
    axes[2].set_ylabel("Temperature\n[K]")

    axes[3].plot(frame.index, frame["Bmag"], color="black", label="|B|", linewidth=1.5)
    for component, color in zip(["Bx_GSE", "By_GSE", "Bz_GSE"], ["tab:blue", "tab:green", "tab:red"]):
        axes[3].plot(frame.index, frame[component], label=component, color=color, alpha=0.8)
    axes[3].axhline(0, color="0.5", linewidth=0.8)
    axes[3].set_ylabel("IMF [nT]")
    axes[3].legend(ncol=4, loc="upper right")

    axes[0].set_title(title)
    axes[-1].set_xlabel("Time [UTC]")
    fig.tight_layout()
    return fig, axes


plot_solar_wind_overview(solar1, "SOLAR-1: most recent 24 hours");

## Exercise 1 — What is the solar wind like today?

Use the overview above and the small toolbox below. Focus on what the measurements show rather than on producing a particular plot.

1. Estimate representative values of speed, density, temperature, and magnetic-field strength during the last 24 hours. How variable is each quantity?
2. Identify one interval in which something changes noticeably. Which quantities change together, and which do not?
3. Compare the beginning and end of the interval. Would you describe the solar wind as steady during this day?
4. Look for gaps or isolated values. How could you distinguish a real rapid change from a data problem?
5. SOLAR-1 is roughly 1.5 million km upstream of Earth. Using a speed from today's data, estimate the travel time from L1 to Earth. How much warning does this provide?
6. Assuming a constant speed, how long has it taken this wind to travel from the Sun?
7. Compare your latest values with NOAA's [live solar-wind display](https://www.spaceweather.gov/products/solar-wind). Check that the selected spacecraft and timestamps agree.

Write down two observations and one question raised by the data.

In [ ]:
# Useful operations — change the columns or time interval as needed.
variables = ["speed", "density", "temperature", "Bmag"]

summary = pd.DataFrame({
    "minimum": solar1[variables].min(),
    "median":  solar1[variables].median(),
    "standard deviation": solar1[variables].std(),
    "maximum": solar1[variables].max(),
})

summary

In [ ]:
# Example: select the final three hours and plot one variable.
last_three_hours = solar1.loc[solar1.index.max() - pd.Timedelta("3h"):]
last_three_hours["speed"].plot(ylabel="Speed [km/s]", title="Final three hours");

### Your observations

- 
- 

**A question raised by the data:** 

# 2. A longer view: hourly OMNI data

A single day cannot tell us what is typical or reveal multi-week patterns. The function below downloads a chosen interval from the hourly OMNI dataset and renames the variables to match the recent-data DataFrame.

The default interval is **2008**, near a deep solar minimum. The downloaded file is cached beside the notebook, so rerunning the cell does not download it again.

> Important: OMNI timestamps have been time-shifted to the bow shock. The recent SOLAR-1 timestamps refer to measurements near L1. This matters for event-by-event comparisons, although it has little effect on the statistical comparisons below.

In [ ]:
HAPI_DATA_URL = "https://cdaweb.gsfc.nasa.gov/hapi/data"
OMNI_DATASET = "OMNI2_H0_MRG1HR"


def download_omni_hourly(start, stop, use_cache=True):
    remote_columns = [
        "BX_GSE1800", "BY_GSE1800", "BZ_GSE1800",
        "BY_GSM1800", "BZ_GSM1800", "N1800", "V1800"
    ]
    local_columns = [
        "time", "Bx_GSE", "By_GSE", "Bz_GSE",
        "By_GSM", "Bz_GSM", "density", "speed"
    ]

    start_label = pd.Timestamp(start).strftime("%Y%m%d")
    stop_label = pd.Timestamp(stop).strftime("%Y%m%d")
    cache_file = Path(f"omni_{start_label}_{stop_label}_hourly.csv")

    if use_cache and cache_file.exists():
        frame = pd.read_csv(cache_file, parse_dates=["time"])
        frame["time"] = pd.to_datetime(frame["time"], utc=True)
        return frame.set_index("time").sort_index()

    query = urlencode({
        "id": OMNI_DATASET,
        "parameters": ",".join(remote_columns),
        "time.min": pd.Timestamp(start, tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
        "time.max": pd.Timestamp(stop, tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
        "format": "csv"
    })

    frame = pd.read_csv(
        f"{HAPI_DATA_URL}?{query}",
        names=local_columns,
        na_values=[999.9, 9999.0, 99999.0, 999999.0, 9999999.0]
    )
    frame["time"] = pd.to_datetime(frame["time"], utc=True)

    for column in local_columns[1:]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["Bmag"] = np.sqrt(frame["Bx_GSE"]**2 + frame["By_GSE"]**2 + frame["Bz_GSE"]**2)

    if use_cache:
        frame.to_csv(cache_file, index=False)

    return frame.set_index("time").sort_index()

In [ ]:
LONG_START = "2008-01-01"
LONG_STOP = "2009-01-01"

omni = download_omni_hourly(LONG_START, LONG_STOP)
print(f"Downloaded {len(omni):,} hourly samples from {omni.index.min()} to {omni.index.max()}")
omni.head()

## Exercise 2 — What is typical?

1. Inspect the distributions of speed, density, and magnetic-field strength. Are they symmetric, or do they have long tails?
2. Find useful ranges containing most of the observations—for example, the 5th to 95th percentiles.
3. Are the mean and median similar? Which one would you quote as a typical value, and why?
4. Look at one or two extreme values in context. Do they appear to be brief spikes, sustained intervals, or missing-data codes that escaped the cleaning?
5. Compare the most recent 24 hours with the 2008 distribution. Which quantity looks most unusual today?

In [ ]:
long_variables = ["speed", "density", "Bmag"]

omni[long_variables].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, column, unit in zip(axes, long_variables, ["km/s", "cm$^{-3}$", "nT"]):
    axis.hist(omni[column].dropna(), bins=50)
    axis.set(xlabel=f"{column} [{unit}]", ylabel="Number of hours")
fig.tight_layout();

In [ ]:
comparison = omni[long_variables].quantile([0.05, 0.5, 0.95]).T
comparison.columns = ["OMNI 5th percentile", "OMNI median", "OMNI 95th percentile"]
comparison["recent 24-hour median"] = solar1[long_variables].median()
comparison

# 3. Does the IMF have a preferred direction?

In geocentric solar ecliptic (**GSE**) coordinates, the $+x$ axis points from Earth toward the Sun, and the $x$–$y$ plane is the ecliptic plane. If the IMF direction were completely random, the points $(B_x,B_y)$ would have no preferred orientation and the direction angles would be distributed evenly.

We describe the direction in the GSE $x$–$y$ plane by the signed angle

$$
\phi = \operatorname{atan2}(B_y,B_x),
$$

which ranges from −180° to 180°. Keeping the sign and the full angular range allows opposite magnetic-field directions to remain distinguishable.

In [ ]:
omni["phi_GSE"] = np.degrees(np.arctan2(omni["By_GSE"], omni["Bx_GSE"]))

omni[["Bx_GSE", "By_GSE", "phi_GSE"]].head()

In [ ]:
field_limit = omni[["Bx_GSE", "By_GSE"]].abs().quantile(0.99).max()
angle_bins = np.arange(-180, 181, 10)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(omni["Bx_GSE"], omni["By_GSE"], s=4, alpha=0.15)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(
    xlabel="$B_x$ GSE [nT]",
    ylabel="$B_y$ GSE [nT]",
    xlim=(-field_limit, field_limit),
    ylim=(-field_limit, field_limit),
    title="IMF components in the GSE x-y plane",
)
axes[0].set_aspect("equal")

axes[1].hist(omni["phi_GSE"].dropna(), bins=angle_bins)
axes[1].set(
    xlabel=r"IMF direction $\phi$ [degrees]",
    ylabel="Number of hours",
    title="Distribution of IMF direction",
)
axes[1].set_xticks([-180, -90, 0, 90, 180])

fig.tight_layout()

## Exercise 3 — Discovering structure in the IMF direction

1. Does the $(B_x,B_y)$ cloud look circular, or does it have a preferred axis? Sketch or describe that axis.
2. Is the angle distribution uniform? Identify the centres of any broad peaks.
3. Approximately how far apart are the peaks? What does that separation imply about the relationship between the two preferred directions?
4. Compare the preferred axis with the idealized Parker spiral from the lecture. Is the IMF usually radial, perpendicular to the Sun–Earth line, or somewhere between?
5. Inspect the shorter interval below. Does the IMF switch direction randomly every hour, or remain near one preferred direction for longer intervals?
6. Remembering that GSE $+x$ points toward the Sun, decide which peak represents a generally sunward field and which represents a generally antisunward field. These extended intervals are known as **toward** and **away** IMF sectors. Estimate how long a sector typically persists in your chosen interval.
7. Compare slow and fast wind using the final plot. Does the preferred direction change with solar wind speed? Can you guess why there could be a connection between the IMF angle and solar wind speed?



In [ ]:
# Change these dates to inspect another interval.
window = omni.loc["2008-01-01":"2008-03-01"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].scatter(window.index, window["phi_GSE"], s=6)
axes[0].set(ylabel=r"IMF direction $\phi$ [degrees]", ylim=(-180, 180))
axes[0].set_yticks([-180, -90, 0, 90, 180])

axes[1].plot(window.index, window["speed"], color="tab:blue")
axes[1].set(ylabel="Speed [km/s]", xlabel="Time [UTC]")

fig.tight_layout()

In [ ]:
slow_wind = omni.loc[omni["speed"] < 400]
fast_wind = omni.loc[omni["speed"] > 500]

plt.hist(
    slow_wind["phi_GSE"].dropna(),
    bins=angle_bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Slow wind: speed < 400 km/s",
)
plt.hist(
    fast_wind["phi_GSE"].dropna(),
    bins=angle_bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Fast wind: speed > 500 km/s",
)
plt.xlabel(r"IMF direction $\phi$ [degrees]")
plt.ylabel("Probability density")
plt.xticks([-180, -90, 0, 90, 180])
plt.legend()

### Your observations

- 
- 

**What evidence did you find for large-scale sector structure in the IMF?**

# 4. Is there a repeating pattern?

The solar-wind speed varies strongly, but some of its larger features may recur. We will first average the hourly measurements into daily values. This suppresses short fluctuations and makes patterns on longer timescales easier to see.

An autocorrelation compares a time series with shifted copies of itself. For each lag, it measures how closely values in the original series resemble values observed that many days later. Use the plot below to look for a preferred recurrence time without assuming one in advance.

In [ ]:
daily = omni[["speed", "density", "Bmag"]].resample("1D").mean()

daily["speed"].plot(figsize=(12, 4), ylabel="Daily mean speed [km/s]", title="Solar-wind speed during 2008");

In [ ]:
lags = np.arange(1, 61)
speed_autocorrelation = pd.Series(
    [daily["speed"].autocorr(lag=int(lag)) for lag in lags],
    index=lags
)

speed_autocorrelation.plot(marker="o", markersize=3)
plt.xlabel("Lag [days]")
plt.ylabel("Autocorrelation")
plt.title("Recurrence of daily solar-wind speed");

## Exercise 4 — Can you find a recurring timescale?

1. Inspect the daily speed time series before looking at the autocorrelation. Can you find any prominent speed enhancements that appear more than once? Estimate the time between them.
2. Now inspect the autocorrelation. Ignore the high values at the shortest lags, which mainly show that the solar wind changes continuously rather than instantaneously. Where is the strongest peak at a longer lag?
3. Is there another peak near twice this lag? How does that strengthen, or weaken, the evidence for a repeating pattern?
4. Use the next cell to overlay the speed with a shifted copy. Find an interval where the recurrence is clear and one where it breaks down.
5. What process involving the Sun could produce the timescale you found?
6. Repeat the autocorrelation for density or magnetic-field strength. Which quantity shows the clearest recurrence?
7. Change `LONG_START` and `LONG_STOP` to explore another year. Is the same periodicity always equally clear?

Possible years to compare include 2003 (more active) and 2018 (closer to solar minimum).

In [ ]:
# Find the strongest recurrence after the short-lag correlations.
candidate_lag = 0 # insert the peak lag
print(f"Candidate recurrence time: {candidate_lag} days")

comparison_interval = daily.loc["2008-01-01":"2008-06-30", "speed"]
comparison_interval.plot(label="speed", figsize=(12, 4))
comparison_interval.shift(candidate_lag).plot(label=f"shifted by {candidate_lag} days", alpha=0.8)
plt.ylabel("Daily mean speed [km/s]")
plt.legend();

# 5. Choose your own question

Use either dataset to investigate one question of your own. Examples:

- Does high-speed wind tend to have a different density from slow wind?
- Does magnetic-field strength increase during sharp density changes?
- Which quantity is smoothest, and which fluctuates most rapidly?
- Are toward and away sectors equally common during your chosen year?
- Does the 27-day recurrence appear in IMF sector polarity as well as speed?

Make one plot or table and write a short conclusion. Also state one limitation of your analysis.

In [ ]:
# Your exploration

### Conclusion

**Question:**  

**What I found:**  

**One limitation:**  